In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Hand-Pose Confidence Estimation under Occlusion \u2014 GPU run\n",
    "\n",
    "Runs the real experiment: **HaMeR** on **HO-3D v3**, four uncertainty estimators,\n",
    "occlusion-stratified analysis. The analysis pipeline is already validated end-to-end\n",
    "on synthetic data (`scripts/synthetic_validation.py`).\n",
    "\n",
    "**Before running:** Runtime \u2192 Change runtime type \u2192 **GPU** (T4 is fine, A100 faster).\n",
    "\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 0. GPU + Drive\n",
    "!nvidia-smi -L\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1. Get the project code (pick ONE)\n",
    "# Option A: from Drive\n",
    "!cp -r /content/drive/MyDrive/cs231n-project /content/proj\n",
    "# Option B: from GitHub (if you pushed it)\n",
    "# !git clone https://github.com/<you>/cs231n-project /content/proj\n",
    "%cd /content/proj\n",
    "!pip -q install -r requirements.txt\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 2. Install HaMeR + checkpoints (~10 min first time; cache to Drive after)\n",
    "%cd /content\n",
    "!git clone --recursive https://github.com/geopavlakos/hamer.git\n",
    "%cd hamer\n",
    "!pip -q install -e .[all]\n",
    "!pip -q install -v -e third-party/ViTPose\n",
    "# download trained models (~2.5GB). If this URL changes, see the HaMeR README.\n",
    "!bash fetch_demo_data.sh\n",
    "%cd /content/proj\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 3. Smoke test: HaMeR loads and runs on one image\n",
    "import sys; sys.path.insert(0, '/content/proj')\n",
    "from src.hamer_wrapper import HamerPredictor\n",
    "import numpy as np, cv2\n",
    "pred = HamerPredictor()\n",
    "print('HaMeR loaded OK')\n",
    "# NOTE: if load_hamer import fails, check hamer's current API \u2014\n",
    "# adjust the import at the top of src/hamer_wrapper.py accordingly.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 4. Point at HO-3D on Drive\n",
    "HO3D = '/content/drive/MyDrive/HO3D_v3'\n",
    "from src.ho3d_data import HO3D as HO3DDataset\n",
    "ds = HO3DDataset(HO3D, split='train')\n",
    "print(f'{len(ds)} frames available')\n",
    "fr = ds[0]\n",
    "print('first frame:', fr.seq, fr.idx, fr.image_path.exists())\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 5. PILOT: 500 frames end-to-end (expect ~20-40 min on T4)\n",
    "!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_pilot \\\n",
    "    --stage all --max-frames 500 --tta 4\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 6. Inspect pilot report\n",
    "import json\n",
    "print(json.dumps(json.load(open('/content/drive/MyDrive/results_pilot/report.json')), indent=2))\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 7. FULL RUN (start with 10-20k frames; results cache to Drive, resumable)\n",
    "!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_full \\\n",
    "    --stage all --max-frames 20000 --tta 8\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 8. Figures\n",
    "from src import plots\n",
    "plots.make_all('/content/drive/MyDrive/results_full/scores.npz',\n",
    "               '/content/drive/MyDrive/results_full/figures')\n",
    "from IPython.display import Image, display\n",
    "for f in ['sparsification','occlusion_stratified','filtering','score_vs_error']:\n",
    "    display(Image(f'/content/drive/MyDrive/results_full/figures/{f}.png'))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Notes\n",
    "- **Occlusion masks**: `run_eval.py` reads per-frame masks from `results*/masks/`.\n",
    "  Render them with `src/ho3d_data.amodal_object_mask` (needs YCB object meshes from the\n",
    "  HO-3D README) \u2014 without them, everything still runs but occlusion stratification is skipped.\n",
    "- **Resumable**: inference caches per-frame `.npz`; re-running skips finished frames.\n",
    "- **Colab disconnects**: results live on Drive, so nothing is lost.\n"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}